# INV-M365-CF Authoritative Persona Post-H5 Department-Pack Coverage Parity Correction v1

## Purpose
- Prove the fresh `M1` replay remains blocked because the department-pack authority surface still encodes staged pre-activation coverage for the 20 promoted personas.

## Lemma Mapping
- `L84`: A fresh merge replay to `development` cannot pass the department-pack validation slice until every affected department-pack persona coverage status reconciles to the live authoritative persona registry.

## Invariant Support
- Fail closed when any department-pack persona declares a `coverage_status` that differs from `registry/persona_registry_v2.yaml`.
- Require an explicit affected-pack inventory before widening the next governed correction package.

## Expected Results
- A deterministic verification artifact records the fresh blocked merge commit, the corrected source branch head, the full mismatch inventory, and the minimum package scope required before `M1` may be replayed.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import yaml

repo_root = Path.cwd()
registry_path = repo_root / "registry" / "persona_registry_v2.yaml"
department_pack_paths = sorted((repo_root / "registry").glob("department_pack_*_v1.yaml"))
output_path = (
    repo_root
    / "configs"
    / "generated"
    / "authoritative_persona_post_h5_department_pack_coverage_parity_correction_v1_verification.json"
)

registry_payload = yaml.safe_load(registry_path.read_text(encoding="utf-8"))
registry_personas = registry_payload["personas"]

affected_departments = []
mismatch_total = 0
for pack_path in department_pack_paths:
    pack_payload = yaml.safe_load(pack_path.read_text(encoding="utf-8"))
    department_id = pack_payload["department"]["id"]
    mismatches = []
    for persona_id, declared in (pack_payload.get("personas") or {}).items():
        registry_persona = registry_personas[persona_id]
        declared_status = declared["coverage_status"]
        registry_status = registry_persona["coverage_status"]
        if declared_status != registry_status:
            mismatches.append(
                {
                    "persona_id": persona_id,
                    "declared_coverage_status": declared_status,
                    "registry_coverage_status": registry_status,
                    "declared_supported_action_count": len(declared.get("supported_actions") or []),
                    "registry_allowed_action_count": len(registry_persona.get("allowed_actions") or []),
                }
            )
    if mismatches:
        mismatch_total += len(mismatches)
        affected_departments.append(
            {
                "department_id": department_id,
                "pack_path": str(pack_path.relative_to(repo_root)),
                "mismatch_count": len(mismatches),
                "mismatches": mismatches,
            }
        )

payload = {
    "lemma_id": "L84",
    "fresh_m1_blocked_merge_commit": "d194b94",
    "fresh_m1_blocked_merge_branch": "development",
    "fresh_m1_blocked_merge_pushed": False,
    "fresh_m1_blocked_merge_safety_branch": "codex/m1-fresh-replay-blocked-development-d194b94",
    "previous_blocked_merge_safety_branch": "codex/m1-replay-blocked-development-a895678",
    "source_branch": "codex/m365-authoritative-persona-post-h5-parity-correction",
    "source_branch_head": "e85be90",
    "origin_development_head": "4c997ea",
    "blocked_by_m1_department_pack_validation": True,
    "affected_department_count": len(affected_departments),
    "mismatch_total": mismatch_total,
    "affected_departments": affected_departments,
    "required_scope": {
        "department_pack_authorities": [item["pack_path"] for item in affected_departments],
        "commercialization_contracts": [
            "docs/commercialization/m365-communication-department-pack-v1.md",
            "docs/commercialization/m365-engineering-department-pack-v1.md",
            "docs/commercialization/m365-hr-department-pack-v1.md",
            "docs/commercialization/m365-marketing-department-pack-v1.md",
            "docs/commercialization/m365-operations-department-pack-v1.md",
            "docs/commercialization/m365-project-management-department-pack-v1.md",
            "docs/commercialization/m365-studio-operations-department-pack-v1.md",
        ],
        "verifiers": [
            "scripts/ci/verify_communication_department_pack_v1.py",
            "scripts/ci/verify_engineering_department_pack_v1.py",
            "scripts/ci/verify_hr_department_pack_v1.py",
            "scripts/ci/verify_marketing_department_pack_v1.py",
            "scripts/ci/verify_operations_department_pack_v1.py",
            "scripts/ci/verify_project_management_department_pack_v1.py",
            "scripts/ci/verify_studio_operations_department_pack_v1.py",
        ],
        "tests": [
            "tests/test_communication_department_pack_v1.py",
            "tests/test_engineering_department_pack_v1.py",
            "tests/test_hr_department_pack_v1.py",
            "tests/test_marketing_department_pack_v1.py",
            "tests/test_operations_department_pack_v1.py",
            "tests/test_project_management_department_pack_v1.py",
            "tests/test_studio_operations_department_pack_v1.py",
        ],
    },
}

output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps({"lemma_id": payload["lemma_id"], "mismatch_total": mismatch_total, "affected_department_count": len(affected_departments)}, indent=2, sort_keys=True))
